In [ ]:
import datasets
import pandas as pd
import shap
import transformers

shap.initjs()

# load the emotion dataset
dataset = datasets.load_dataset("emotion", split="train")
data = pd.DataFrame({"text": dataset["text"], "emotion": dataset["label"]})

In [ ]:
from tklearn.kb import KnowledgeBase

kb = KnowledgeBase("wiktionary")

In [ ]:
# load the model and tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(
    "nateraw/bert-base-uncased-emotion", use_fast=True
)
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    "nateraw/bert-base-uncased-emotion"
).to("mps")

# build a pipeline object to do predictions
pipeline = transformers.pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device="mps",
    top_k=None,
)

In [ ]:
explainer = shap.Explainer(pipeline)

In [ ]:
from random import Random

import pandas as pd
from tqdm import auto as tqdm

random = Random(42)

preds = []

batch_size = 32
shap_values = None
batch = {
    "text": [],
    "augment": [],
    "idx": [],
}
NUM_AUGMENTS = 3

pbar = tqdm.trange(data.shape[0], desc="Processing", leave=True)

for idx in pbar:
    original_text = data["text"][idx]
    augments = list(kb.augment(original_text, mentions=None))
    # take a random sample of mentions
    if len(augments) > NUM_AUGMENTS:
        augments = augments[:1] + random.sample(augments[1:], NUM_AUGMENTS)
    for augment in augments:
        text = augment["text"]
        batch["text"].append(text)
        batch["augment"].append(augment)
        batch["idx"].append(idx)
        if len(batch["text"]) < batch_size:
            continue
        shap_values = explainer(batch["text"])  # slow
        for i in range(len(batch["text"])):
            scores = shap_values.values[i].mean(0)
            labels = [f"class_{c}" for c in range(len(scores))]
            preds.append({
                "idx": batch["idx"][i],
                **batch["augment"][i],
                "labels": {labels[j]: scores[j] for j in range(len(scores))},
            })
        batch = {
            "text": [],
            "augment": [],
            "idx": [],
        }
        pbar.set_postfix({
            "num_preds": len(preds),
        })
    # break
    # if shap_values is not None:
    break

In [ ]:
from datasets import Dataset

df = pd.DataFrame(preds)

df.to_csv("emotion_bert_shap.csv", index=False)

In [ ]:
df.head()

In [ ]:
df[df["support"] == 0]

In [ ]:
labels_df = df["labels"].apply(pd.Series)

labels_cols = labels_df.columns.tolist()

df = pd.concat([df, labels_df], axis=1)

In [ ]:
df.head(1)

In [ ]:
from scipy.spatial.distance import cdist

In [ ]:
df["score"] = 0.0

for i in df.idx.unique():
    original_labels_df = labels_df.loc[
        (df.idx == i) & (df.support == 0), labels_cols
    ]
    augmented_labels_df = labels_df.loc[
        (df.idx == i) & (df.support != 0), labels_cols
    ]
    d = cdist(augmented_labels_df, original_labels_df, metric="cosine")
    scores = (1 - d[:, 0]).round(2)
    df.loc[(df.idx == i) & (df.support != 0), "score"] = scores

In [ ]:
df

In [ ]:
temp_df = (
    df[["relations", "score"]]
    .explode("relations")
    .dropna()
    .groupby("relations")
    .mean()
    .reset_index()
    .sort_values("score", ascending=False)
)

In [ ]:
# select top 10 and bottom 10 relations
top_relations = temp_df.head(20)
bottom_relations = temp_df.tail(20)
pd.concat([top_relations, bottom_relations]).reset_index(drop=True)